# DJIA Data Exploration

Explore the downloaded DJIA stock data and computed technical indicators.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_feather('data/processed/djia_processed.feather')
print(f'Shape: {df.shape}')
print(f'Tickers: {sorted(df.ticker.unique())}')
print(f'Date range: {df.timestamp.min()} to {df.timestamp.max()}')
df.head()

## Missing Values

In [ ]:
print('Missing values per column:')
print(df.isna().sum())
print(f'\nTotal NaN: {df.isna().sum().sum()}')
print(f'\nRows per ticker:')
print(df.groupby('ticker').size())

## Basic Statistics

In [ ]:
numeric_cols = ['open', 'high', 'low', 'close', 'volume', 'rsi_14', 'macd', 'daily_return']
df[numeric_cols].describe()

## Price Charts (Selected Stocks)

In [ ]:
sample_tickers = ['AAPL', 'MSFT', 'JPM', 'JNJ']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, ticker in zip(axes.flat, sample_tickers):
    tdf = df[df.ticker == ticker]
    ax.plot(tdf.timestamp, tdf.close, linewidth=0.8)
    ax.set_title(ticker)
    ax.set_ylabel('Close Price')
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('DJIA Stock Prices (2009-2022)', fontsize=14)
plt.tight_layout()
plt.savefig('reports/cycle_2/price_charts.png', dpi=100, bbox_inches='tight')
plt.show()

## Technical Indicators (AAPL)

In [ ]:
aapl = df[df.ticker == 'AAPL'].copy()

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# Price with Bollinger Bands
axes[0].plot(aapl.timestamp, aapl.close, label='Close', linewidth=0.8)
axes[0].plot(aapl.timestamp, aapl.bb_upper, '--', alpha=0.5, label='BB Upper')
axes[0].plot(aapl.timestamp, aapl.bb_lower, '--', alpha=0.5, label='BB Lower')
axes[0].fill_between(aapl.timestamp, aapl.bb_lower, aapl.bb_upper, alpha=0.1)
axes[0].set_title('AAPL - Price & Bollinger Bands')
axes[0].legend()

# RSI
axes[1].plot(aapl.timestamp, aapl.rsi_14, linewidth=0.8, color='purple')
axes[1].axhline(70, color='red', linestyle='--', alpha=0.5)
axes[1].axhline(30, color='green', linestyle='--', alpha=0.5)
axes[1].set_title('RSI(14)')
axes[1].set_ylim(0, 100)

# MACD
axes[2].plot(aapl.timestamp, aapl.macd, label='MACD', linewidth=0.8)
axes[2].plot(aapl.timestamp, aapl.macd_signal, label='Signal', linewidth=0.8)
axes[2].bar(aapl.timestamp, aapl.macd_hist, alpha=0.3, width=1, label='Histogram')
axes[2].set_title('MACD(12,26,9)')
axes[2].legend()

# Volume
axes[3].bar(aapl.timestamp, aapl.volume, alpha=0.5, width=1)
axes[3].set_title('Volume')

plt.suptitle('AAPL Technical Indicators', fontsize=14)
plt.tight_layout()
plt.savefig('reports/cycle_2/technical_indicators.png', dpi=100, bbox_inches='tight')
plt.show()

## Daily Return Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for ticker in sample_tickers:
    tdf = df[df.ticker == ticker]
    ax.hist(tdf.daily_return, bins=100, alpha=0.4, label=ticker, density=True)
ax.set_title('Daily Return Distribution')
ax.set_xlabel('Daily Return')
ax.legend()
plt.tight_layout()
plt.savefig('reports/cycle_2/return_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

## Correlation Matrix

In [ ]:
# Pivot to get daily returns per ticker
returns = df.pivot_table(index='timestamp', columns='ticker', values='daily_return')
corr = returns.corr()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)
plt.colorbar(im)
ax.set_title('Daily Return Correlation Matrix')
plt.tight_layout()
plt.savefig('reports/cycle_2/correlation_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'Average pairwise correlation: {corr.values[np.triu_indices_from(corr.values, k=1)].mean():.3f}')